In [2]:
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [4]:
COLLECTION_NAME = "llama2-research-paper"
path = r"F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Vector_database\FAISS\2307.09288v2.pdf"

loader = PyPDFLoader(path)
pages = loader.load()

print("PDF pages loaded:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


PDF pages loaded: 77


In [5]:
# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 175


In [6]:
# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

In [8]:
# ---------------------------------------------------
# 4. Embedding model
# ---------------------------------------------------
import os
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-3-large",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

vector = embeddings.embed_query("What is Retrieval Augmented Generation?")

print("Embedding dimensions:", len(vector))

Embedding dimensions: 3072


In [9]:
# # 5. Local Qdrant
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )

In [11]:
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")

client = QdrantClient(
    api_key=qdrant_api_key, 
    url=qdrant_cluster_endpoint
)

In [13]:
# 6. Create collection
dimension = 3072
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

In [14]:
# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

In [15]:
# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

In [16]:
# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 175


In [17]:
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: approve it. If the answer could not be approved without major changes, the reviewers were asked to reject it
and write the feedback necessary to improve it.
A.5.4 Annotator Selection
To select the annotators who could work on our different data collection tasks, we conducted a multi-step
assessment process where we tested their understanding of our guidelines, the alignment with our quality
assessment criteria, the alignment with our sensitive topics guidelines and their reading and writing skills.
The process included 4 tests:
• Thefirsttestconsistsof3sectionsoftestingtoevaluategrammar,readingcomprehensionandwriting
style. Each section is timed and the test should take a total of 50 minutes to complete. A candidate
must score 90% on part I to continue on to parts II and III, and an average score of 4 on part II and III
to pass the test.
• The second test consisted of 42 questions split into sensitive topics alignment, answer ranking and
two examples of answer writin

In [18]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

YaGuang Li, Jian Li, Hyeontaek Lim, Hanzhao Lin, Zhongtao Liu, Frederick Liu, Marcello Maggioni,
Aroma Mahendru, Joshua Maynez, Vedant Misra, Maysam Moussalem, Zachary Nado, John Nham, Eric
Ni, Andrew Nystrom, Alicia Parrish, Marie Pellat, Martin Polacek, Alex Polozov, Reiner Pope, Siyuan Qiao,
Emily Reif, Bryan Richter, Parker Riley, Alex Castro Ros, Aurko Roy, Brennan Saeta, Rajkumar Samuel,
Renee Shelby, Ambrose Slone, Daniel Smilkov, David R. So, Daniel Sohn, Simon Tokumine, Dasha Valter,
Vijay Vasudevan, Kiran Vodrahalli, Xuezhi Wang, Pidong Wang, Zirui Wang, Tao Wang, John Wieting,
Yuhuai Wu, Kelvin Xu, Yunhan Xu, Linting Xue, Pengcheng Yin, Jiahui Yu, Qiao Zhang, Steven Zheng,
Ce Zheng, Weikang Zhou, Denny Zhou, Slav Petrov, and Yonghui Wu. Palm 2 technical report, 2023.
Amanda Askell, Yuntao Bai, Anna Chen, Dawn Drain, Deep Ganguli, Tom Henighan, Andy Jones, Nicholas
Joseph, Ben Mann, Nova DasSarma, Nelson Elhage, Zac Hatfield-Dodds, Danny Hernandez, Jackson
Kernion, Kamal Ndou